In [2]:
import sympy as sp
from sympy import symbols, diff, solve

In [3]:
# Define Symbols
psi_dot, m, p, C_d, A, V_ref, V_y, F_x, e_vx, e_psi, e_y = symbols(
    "psi_dot m p C_d A V_ref V_y F_x e_vx e_psi e_y"
)
C_alp_f, C_alp_r, L_f, L_r, delta, I2, K = symbols("C_alp_f C_alp_r L_f L_r delta I2 K")

# State & Control Vectors
x = sp.Matrix([e_vx, V_y, psi_dot, e_y, e_psi]) 
u = sp.Matrix([F_x, delta]) 

# Longitudinal Velocity Error Dynamics
f1 = psi_dot * V_y + 1 / m * (F_x - 0.5 * p * C_d * A * (V_ref + e_vx) ** 2)

# Lateral Velocity Error Dynamics (Removed trailing *)
f2 = -psi_dot * (V_ref + e_vx) + 1 / m * (
    -C_alp_f * ((V_y + L_f * psi_dot) / (V_ref + e_vx) - delta)
    - C_alp_r * ((V_y - L_r * psi_dot) / (V_ref + e_vx))
)

# Yaw Rate Error Dynamics (Removed trailing * and fixed parenthetical grouping)
f3 = 1 / I2 * (
    L_f * -C_alp_f * ((V_y + L_f * psi_dot) / (V_ref + e_vx) - delta)
    - L_r * -C_alp_r * ((V_y - L_r * psi_dot) / (V_ref + e_vx))
)

# CTE Kinematics
f4 = (V_ref + e_vx) * sp.sin(e_psi) + V_y * sp.cos(e_psi)

# Heading Error Kinematics
f5 = psi_dot - K * (e_vx + V_ref)

# State Dynamics Vector
f = sp.Matrix([f1, f2, f3, f4, f5])

In [4]:
# Compute Jacobians
A = f.jacobian(x)
B = f.jacobian(u)
# Display Results
print("A Matrix (State Jacobian):")
sp.pprint(A)
print("\nB Matrix (Control Jacobian):")
sp.pprint(B)


A Matrix (State Jacobian):
⎡               -0.5⋅A⋅C_d⋅p⋅(2⋅V_ref + 2⋅eᵥₓ)                                 ↪
⎢               ───────────────────────────────                            ψ_d ↪
⎢                              m                                               ↪
⎢                                                                              ↪
⎢         C_alp_f⋅(L_f⋅ψ_dot + V_y)   Cₐₗₚ ᵣ⋅(-Lᵣ⋅ψ_dot + V_y)                 ↪
⎢         ───────────────────────── + ────────────────────────      C_alp_f    ↪
⎢                           2                           2       - ───────────  ↪
⎢              (V_ref + eᵥₓ)               (V_ref + eᵥₓ)          V_ref + eᵥₓ  ↪
⎢-ψ_dot + ────────────────────────────────────────────────────  ────────────── ↪
⎢                                  m                                         m ↪
⎢                                                                              ↪
⎢ C_alp_f⋅L_f⋅(L_f⋅ψ_dot + V_y)   Cₐₗₚ ᵣ⋅Lᵣ⋅(-Lᵣ⋅ψ_dot + V_y)                  ↪
⎢

In [5]:
# Evaluate at equilibrium point (e_vx=0, V_y=0, psi_dot=0, e_y=0, e_psi=0, F_x=0, delta=0)
equilibrium_point = {
    e_vx: 0,
    V_y: 0,
    psi_dot: 0,
    e_y: 0,
    e_psi: 0,
    F_x: 0,
    delta: 0,
}
A_eq = A.subs(equilibrium_point)
B_eq = B.subs(equilibrium_point)
print("\nA Matrix at Equilibrium:")
sp.pprint(A_eq)
print("\nB Matrix at Equilibrium:")
sp.pprint(B_eq)


A Matrix at Equilibrium:
⎡-1.0⋅A⋅C_d⋅V_ref⋅p                                                            ↪
⎢───────────────────              0                              0             ↪
⎢         m                                                                    ↪
⎢                                                                              ↪
⎢                          C_alp_f   Cₐₗₚ ᵣ                 C_alp_f⋅L_f   Cₐₗₚ ↪
⎢                        - ─────── - ──────               - ─────────── + ──── ↪
⎢                           V_ref    V_ref                     V_ref        V_ ↪
⎢         0              ──────────────────      -V_ref + ──────────────────── ↪
⎢                                m                                    m        ↪
⎢                                                                              ↪
⎢                                                                2             ↪
⎢                       C_alp_f⋅L_f   Cₐₗₚ ᵣ⋅Lᵣ       C_alp_f⋅L_f    Cₐₗₚ ᵣ⋅Lᵣ ↪
⎢ 

## Skidpad Event

In [6]:
# 1. Define our known Skidpad equilibrium constraints
# We know e_vx is 0, so we solve f5 to find the required steady-state yaw rate
psi_dot_eq = sp.solve(f5.subs(e_vx, 0), psi_dot)[0]

# 2. Substitute those knowns (e_vx = 0 and psi_dot_eq) into f2 and f3
f2_corner = f2.subs({e_vx: 0, psi_dot: psi_dot_eq})
f3_corner = f3.subs({e_vx: 0, psi_dot: psi_dot_eq})

# 3. Solve the remaining system simultaneously for V_y and delta
# This returns a dictionary of the solutions
corner_sols = sp.solve([f2_corner, f3_corner], [V_y, delta])

# 4. Safely extract the results into brand new variables
V_y_eq = sp.simplify(corner_sols[V_y])
delta_eq = sp.simplify(corner_sols[delta])

# Display the final expressions
print("Steady-State Yaw Rate (psi_dot_eq):")
sp.pprint(psi_dot_eq)

print("\nSteady-State Lateral Velocity (V_y_eq):")
sp.pprint(V_y_eq)

print("\nSteady-State Steering Angle (delta_eq):")
sp.pprint(delta_eq)

Steady-State Yaw Rate (psi_dot_eq):
K⋅V_ref

Steady-State Lateral Velocity (V_y_eq):
        ⎛                         2            2  ⎞
K⋅V_ref⋅⎝Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + Cₐₗₚ ᵣ⋅Lᵣ  - L_f⋅V_ref ⋅m⎠
───────────────────────────────────────────────────
                 Cₐₗₚ ᵣ⋅(L_f + Lᵣ)                 

Steady-State Steering Angle (delta_eq):
  ⎛                  2                                              2          ↪
K⋅⎝C_alp_f⋅Cₐₗₚ ᵣ⋅L_f  + 2⋅C_alp_f⋅Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + C_alp_f⋅Cₐₗₚ ᵣ⋅Lᵣ  - C_alp_ ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                            C_alp_f⋅Cₐₗₚ ᵣ⋅(L_f + Lᵣ)          ↪

↪            2                    2  ⎞
↪ f⋅L_f⋅V_ref ⋅m + Cₐₗₚ ᵣ⋅Lᵣ⋅V_ref ⋅m⎠
↪ ────────────────────────────────────
↪                                     


In [7]:
skidpad_equilibrium = {
    e_vx: 0,
    psi_dot: psi_dot_eq,
    V_y: V_y_eq,
    delta: delta_eq,
    e_y: 0,
    e_psi: 0,
}

A_skidpad = sp.simplify(A.subs(skidpad_equilibrium))
B_skidpad = sp.simplify(B.subs(skidpad_equilibrium)) 

print("\nA Matrix at Skidpad Equilibrium:")
sp.pprint(A_skidpad)    
print("\nB Matrix at Skidpad Equilibrium:")
sp.pprint(B_skidpad)  



A Matrix at Skidpad Equilibrium:
⎡                                                                              ↪
⎢                                                           -1.0⋅A⋅C_d⋅V_ref⋅p ↪
⎢                                                           ────────────────── ↪
⎢                                                                    m         ↪
⎢                                                                              ↪
⎢  ⎛                  2                                              2         ↪
⎢K⋅⎝C_alp_f⋅Cₐₗₚ ᵣ⋅L_f  + 2⋅C_alp_f⋅Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + C_alp_f⋅Cₐₗₚ ᵣ⋅Lᵣ  - C_alp ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                        Cₐₗₚ ᵣ⋅V_ref⋅m⋅(L_f + ↪
⎢                                                                              ↪
⎢                ⎛                  2                                          ↪
⎢          K⋅L_f⋅⎝C_alp_f⋅Cₐₗₚ ᵣ⋅L_f  + 2⋅C_alp_f⋅Cₐₗₚ ᵣ⋅L_f⋅Lᵣ + C_alp_f⋅C

In [8]:
# Write everything to a .tex file for LaTeX rendering
with open("skidpad_equilibrium.tex", "w") as f:
    f.write("% Skidpad Equilibrium Analysis\n")
    f.write("\\documentclass{article}\n")
    f.write("\\usepackage{amsmath}\n")
    f.write("\\usepackage{amssymb}\n")
    f.write("\\begin{document}\n")
    
    f.write("\\section*{A and B Matrices}\n")
    f.write("\\subsection*{A Matrix}\n")
    f.write(f"\\begin{{equation}}\nA = {sp.latex(A)}\n\\end{{equation}}\n")
    f.write("\\subsection*{B Matrix}\n")
    f.write(f"\\begin{{equation}}\nB = {sp.latex(B)}\n\\end{{equation}}\n") 
    
    f.write("\\section*{Skidpad Equilibrium Analysis}\n")
    
    f.write("\\subsection*{Steady-State Yaw Rate (\\(\\psi_{dot,eq}\\))}\n")
    f.write(f"\\begin{{equation}}\n\\psi_{{dot,eq}} = {sp.latex(psi_dot_eq)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{Steady-State Lateral Velocity (\\(V_{y,eq}\\))}\n")
    f.write(f"\\begin{{equation}}\nV_{{y,eq}} = {sp.latex(V_y_eq)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{Steady-State Steering Angle (\\(\\delta_{eq}\\))}\n")
    f.write(f"\\begin{{equation}}\n\\delta_{{eq}} = {sp.latex(delta_eq)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{A Matrix at Skidpad Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nA_{{skidpad}} = {sp.latex(A_skidpad)}\n\\end{{equation}}\n")
    
    f.write("\\subsection*{B Matrix at Skidpad Equilibrium}\n")
    f.write(f"\\begin{{equation}}\nB_{{skidpad}} = {sp.latex(B_skidpad)}\n\\end{{equation}}\n")
    
    f.write("\\end{document}\n")

In [9]:
# Calculate A_straight and B_straight at the straight-line equilibrium point
A_straight = sp.simplify(A_eq.subs(K, 0))  # Set K to 0 for straight-line equilibrium
B_straight = sp.simplify(B_eq.subs(K, 0))  # Set K to 0 for straight-line equilibrium

print("\nA Matrix at Straight-Line Equilibrium:")
sp.pprint(A_straight)
print("\nB Matrix at Straight-Line Equilibrium:")
sp.pprint(B_straight)


A Matrix at Straight-Line Equilibrium:
⎡-1.0⋅A⋅C_d⋅V_ref⋅p                                                            ↪
⎢───────────────────             0                               0             ↪
⎢         m                                                                    ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                        -C_alp_f - Cₐₗₚ ᵣ      -C_alp_f⋅L_f + Cₐₗₚ ᵣ⋅Lᵣ - V_r ↪
⎢         0              ─────────────────      ────────────────────────────── ↪
⎢                             V_ref⋅m                         V_ref⋅m          ↪
⎢                                                                              ↪
⎢                                                                2             ↪
⎢                     -C_alp_f⋅L_f + Cₐₗₚ ᵣ⋅Lᵣ      - C_alp_f⋅L_f  - Cₐₗₚ ᵣ⋅Lᵣ ↪
⎢         0           ────────────────────────      ─────────────────

## Controllability 

In [10]:
## Construct controllability matrix at straight-line equilibrium
C_straight = sp.Matrix.hstack(B_straight,
                            A_straight * B_straight,
                            A_straight**2 * B_straight,
                            A_straight**3 * B_straight,
                            A_straight**4 * B_straight)
C_straight_rank = C_straight.rank()
print("\nControllability Matrix at Straight-Line Equilibrium:")
sp.pprint(C_straight)
print(f"\nRank of Controllability Matrix: {C_straight_rank}")   

## Construct controllability matrix at skidpad equilibrium

## We need to set values to m, V, C_alpha, etc to get a numerical rank
m_val = 1000  # kg
V_val = 20    # m/s
C_alp_f_val = 5   # Front cornering stiffness (N/rad)
C_alp_r_val = 5   # Rear cornering stiffness (N/rad)
L_f_val = 1.5    # Distance from CG to front axle (m)
L_r_val = 1.5    # Distance from CG to rear axle (m)
I2_val = 1500   # Yaw moment of inertia (kg*m^2)
K_val = 0.1    # Path curvature (1/m)
p_val = 1.225  # Air density (kg/m^3)
C_d_val = 0.3   # Drag coefficient
A_val = 2.0     # Frontal area (m^2)

# Substitute values into A_skidpad and B_skidpad
A_skidpad_sub = A_skidpad.subs([(m, m_val), (V_ref, V_val), (C_alp_f, C_alp_f_val),
                                (C_alp_r, C_alp_r_val), (L_f, L_f_val), (L_r, L_r_val),
                                (I2, I2_val), (K, K_val), (p, p_val), (C_d, C_d_val), (A, A_val)])
B_skidpad_sub = B_skidpad.subs([(m, m_val), (V_ref, V_val), (C_alp_f, C_alp_f_val),
                                (C_alp_r, C_alp_r_val), (L_f, L_f_val), (L_r, L_r_val),
                                (I2, I2_val), (K, K_val), (p, p_val), (C_d, C_d_val), (A, A_val)])

# Construct controllability matrix at skidpad equilibrium
C_skidpad = sp.Matrix.hstack(B_skidpad_sub,
                            A_skidpad_sub * B_skidpad_sub,
                            A_skidpad_sub**2 * B_skidpad_sub,
                            A_skidpad_sub**3 * B_skidpad_sub,
                            A_skidpad_sub**4 * B_skidpad_sub)
C_skidpad_rank = C_skidpad.rank()
print("\nControllability Matrix at Skidpad Equilibrium:")
sp.pprint(C_skidpad)
print(f"\nRank of Controllability Matrix: {C_skidpad_rank}")


Controllability Matrix at Straight-Line Equilibrium:
⎡                                                                              ↪
⎢1               -1.0⋅A⋅C_d⋅V_ref⋅p                                            ↪
⎢─       0       ───────────────────                                         0 ↪
⎢m                        2                                                    ↪
⎢                        m                                                     ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢     C_alp_f                         C_alp_f⋅(-C_alp_f - Cₐₗₚ ᵣ)   C_alp_f⋅L_ ↪
⎢0    ───────             0           ─────────────────────────── + ────────── ↪
⎢        m                                            2                        ↪
⎢                                              V_ref⋅m                         ↪
⎢                                                      

## Observability

In [ ]:
## Define the output vector for observability analysis
y_obs = sp.Matrix([e_vx, psi_dot, e_y, e_psi])  # Observing longitudinal velocity error, lateral position error, heading error, and lateral velocity error
C_obs = sp.Matrix([[1, 0, 0, 0, 0],
                  [0, 0, 1, 0, 0],
                  [0, 0, 0, 1, 0],
                  [0, 0, 0, 0, 1]])
D_obs = sp.zeros(4, 2)  # Assuming no direct feedthrough from control to output

## Create the observability matrix at straight-line equilibrium
O_straight = sp.Matrix.vstack(C_obs,
                            C_obs * A_straight,
                            C_obs * A_straight**2,
                            C_obs * A_straight**3,
                            C_obs * A_straight**4)
O_straight_rank = O_straight.rank()
print("\nObservability Matrix at Straight-Line Equilibrium:")
sp.pprint(O_straight)
print(f"\nRank of Observability Matrix: {O_straight_rank}") 

## Create the observability matrix at skidpad equilibrium
O_skidpad = sp.Matrix.vstack(C_obs,
                            C_obs * A_skidpad_sub,
                            C_obs * A_skidpad_sub**2,
                            C_obs * A_skidpad_sub**3,
                            C_obs * A_skidpad_sub**4)
O_skidpad_rank = O_skidpad.rank()
print("\nObservability Matrix at Skidpad Equilibrium:")
sp.pprint(O_skidpad)
print(f"\nRank of Observability Matrix: {O_skidpad_rank}")


Observability Matrix at Straight-Line Equilibrium:
